# Практична робота №3
**Тема:** Очищення та перетворення даних

**Виконав:** Онукевич Данило, КН-41

**Набір даних:** продажі кави в торговому автоматі (`index_1.csv`).

Ноутбук має три частини: кроки практичної частини (1-10), індивідуальні завдання (1-7) та додаткове завдання ★.

# Частина 1. Практична частина

## Крок 1. Підготовка середовища та завантаження даних

In [2]:
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/index_1.csv')
df.head()

Mounted at /content/drive


,date,datetime,cash_type,card,money,coffee_name
0,2024-03-01,2024-03-01 10:15:50.520,card,ANON-0000-0000-0001,38.7,Latte
1,2024-03-01,2024-03-01 12:19:22.539,card,ANON-0000-0000-0002,38.7,Hot Chocolate
2,2024-03-01,2024-03-01 12:20:18.089,card,ANON-0000-0000-0002,38.7,Hot Chocolate
3,2024-03-01,2024-03-01 13:46:33.006,card,ANON-0000-0000-0003,28.9,Americano
4,2024-03-01,2024-03-01 13:48:14.626,card,ANON-0000-0000-0004,38.7,Latte


## Крок 2. Пропущені значення
Створюємо штучний DataFrame з пропусками (`None` перетворюється на `NaN`) і підраховуємо їх у кожній колонці.

In [3]:
data_messy = {
    'name': ['Anna', None, 'Bohdan', 'Olena', None],
    'age': [20, 21, None, 22, 24],
    'score': [85.5, 90.0, 78.3, None, 92.1]
}
df_messy = pd.DataFrame(data_messy)
df_messy

,name,age,score
0,Anna,20.0,85.5
1,None,21.0,90.0
2,Bohdan,NaN,78.3
3,Olena,22.0,NaN
4,None,24.0,92.1


In [4]:
print(df_messy.isnull().sum())

name     2
age      1
score    1
dtype: int64


Два способи обробки: видалення рядків без імені (`dropna`) та заповнення віку середнім (`fillna`).

In [5]:
# Видалення рядків з пропусками у колонці 'name'
df_dropped = df_messy.dropna(subset=['name'])
print(df_dropped)

# Заповнення пропусків у колонці 'age' середнім значенням
mean_age = df_messy['age'].mean()
df_filled = df_messy.copy()
df_filled['age'] = df_filled['age'].fillna(mean_age)
print(df_filled)

     name   age  score
0    Anna  20.0   85.5
2  Bohdan   NaN   78.3
3   Olena  22.0    NaN
     name    age  score
0    Anna  20.00   85.5
1    None  21.00   90.0
2  Bohdan  21.75   78.3
3   Olena  22.00    NaN
4    None  24.00   92.1


## Крок 3. Дублікати

In [6]:
data_dup = {
    'id': [1, 2, 3, 2, 3],
    'value': ['A', 'B', 'C', 'B', 'C']
}
df_dup = pd.DataFrame(data_dup)
df_dup

,id,value
0,1,A
1,2,B
2,3,C
3,2,B
4,3,C


In [7]:
print(df_dup.duplicated())
print("Кількість дублікатів:", df_dup.duplicated().sum())

0    False
1    False
2    False
3     True
4     True
dtype: bool
Кількість дублікатів: 2


In [8]:
df_unique = df_dup.drop_duplicates()
print(df_unique)

   id value
0   1     A
1   2     B
2   3     C


Перевіримо й справжній датасет на повні дублікати.

In [9]:
print("Дублікатів у датасеті продажів:", df.duplicated().sum())

Дублікатів у датасеті продажів: 0


## Крок 4. Викиди (метод IQR)

In [10]:
Q1 = df['money'].quantile(0.25)
Q3 = df['money'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f"Q1 = {Q1}, Q3 = {Q3}, IQR = {IQR}")
print(f"Нижня межа: {lower}, Верхня межа: {upper}")

Q1 = 27.92, Q3 = 35.76, IQR = 7.839999999999996
Нижня межа: 16.160000000000007, Верхня межа: 47.519999999999996


In [11]:
outliers = df[(df['money'] < lower) | (df['money'] > upper)]
print(f"Знайдено викидів: {len(outliers)}")

Знайдено викидів: 0


**Висновок:** усі значення `money` лежать у діапазоні від 18.12 до 40.0, тобто в межах 16.16-47.52. Викидів у датасеті немає, видаляти нічого не потрібно.

## Крок 5. Перетворення типів даних
Колонка `datetime` зчитана як текст. Перетворюємо її на `datetime64`.

In [12]:
print(df.dtypes, "\n")

df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')

print(df.dtypes)

date            object
datetime        object
cash_type       object
card            object
money          float64
coffee_name     object
dtype: object 

date                   object
datetime       datetime64[ns]
cash_type              object
card                   object
money                 float64
coffee_name            object
dtype: object


## Крок 6. Обробка рядкових даних

In [13]:
df_text = pd.DataFrame({
    'city': ['  Kyiv  ', 'LVIV', 'Odesa ', 'KHARKIV', ' kyiv '],
    'phone': ['(044) 123-45-67', '(032) 234-56-78', '(048) 345-67-89', '(057) 456-78-90', '(044) 123-45-67']
})
df_text

,city,phone
0,Kyiv,(044) 123-45-67
1,LVIV,(032) 234-56-78
2,Odesa,(048) 345-67-89
3,KHARKIV,(057) 456-78-90
4,kyiv,(044) 123-45-67


In [14]:
df_text['city'] = df_text['city'].str.strip().str.lower()
df_text['phone'] = df_text['phone'].str.replace(r'\(|\)|-', '', regex=True)
print(df_text)

      city        phone
0     kyiv  044 1234567
1     lviv  032 2345678
2    odesa  048 3456789
3  kharkiv  057 4567890
4     kyiv  044 1234567


## Крок 7. Нові ознаки (feature engineering)

In [15]:
df['date_only'] = df['datetime'].dt.date
df['hour'] = df['datetime'].dt.hour
df['day_of_week'] = df['datetime'].dt.dayofweek
df['month'] = df['datetime'].dt.month
print(df[['datetime', 'date_only', 'hour', 'day_of_week', 'month']].head())

                 datetime   date_only  hour  day_of_week  month
0 2024-03-01 10:15:50.520  2024-03-01    10            4      3
1 2024-03-01 12:19:22.539  2024-03-01    12            4      3
2 2024-03-01 12:20:18.089  2024-03-01    12            4      3
3 2024-03-01 13:46:33.006  2024-03-01    13            4      3
4 2024-03-01 13:48:14.626  2024-03-01    13            4      3


## Крок 8. One-hot encoding

In [16]:
df_encoded = pd.get_dummies(df, columns=['coffee_name'])
print(df_encoded.head())

         date                datetime cash_type                 card  money  \
0  2024-03-01 2024-03-01 10:15:50.520      card  ANON-0000-0000-0001   38.7   
1  2024-03-01 2024-03-01 12:19:22.539      card  ANON-0000-0000-0002   38.7   
2  2024-03-01 2024-03-01 12:20:18.089      card  ANON-0000-0000-0002   38.7   
3  2024-03-01 2024-03-01 13:46:33.006      card  ANON-0000-0000-0003   28.9   
4  2024-03-01 2024-03-01 13:48:14.626      card  ANON-0000-0000-0004   38.7   

    date_only  hour  day_of_week  month  coffee_name_Americano  \
0  2024-03-01    10            4      3                  False   
1  2024-03-01    12            4      3                  False   
2  2024-03-01    12            4      3                  False   
3  2024-03-01    13            4      3                   True   
4  2024-03-01    13            4      3                  False   

   coffee_name_Americano with Milk  coffee_name_Cappuccino  coffee_name_Cocoa  \
0                            False             

## Крок 9. Нормалізація та стандартизація

In [17]:
df['money_norm'] = (df['money'] - df['money'].min()) / (df['money'].max() - df['money'].min())
df['money_std'] = (df['money'] - df['money'].mean()) / df['money'].std()

In [18]:
print(df[['money', 'money_norm', 'money_std']].head())
print("Мінімум money_norm:", df['money_norm'].min())
print("Максимум money_norm:", df['money_norm'].max())
print("Середнє money_std:", df['money_std'].mean())
print("Стандартне відхилення money_std:", df['money_std'].std())

   money  money_norm  money_std
0   38.7    0.940585   1.413261
1   38.7    0.940585   1.413261
2   38.7    0.940585   1.413261
3   28.9    0.492687  -0.578639
4   38.7    0.940585   1.413261
Мінімум money_norm: 0.0
Максимум money_norm: 1.0
Середнє money_std: 1.1725127652806934e-17
Стандартне відхилення money_std: 1.0000000000000462


Середнє `money_std` — це дуже мале число (порядку 1e-16), тобто фактично 0. Так буває через обмежену точність обчислень із дробовими числами.

## Крок 10. Збереження очищених даних

In [19]:
final_df = df[['datetime', 'cash_type', 'money', 'coffee_name', 'hour', 'day_of_week', 'month', 'money_norm', 'money_std']]
final_df.to_csv('/content/drive/MyDrive/cleaned_sales.csv', index=False)
final_df.head()

,datetime,cash_type,money,coffee_name,hour,day_of_week,month,money_norm,money_std
0,2024-03-01 10:15:50.520,card,38.7,Latte,10,4,3,0.940585,1.413261
1,2024-03-01 12:19:22.539,card,38.7,Hot Chocolate,12,4,3,0.940585,1.413261
2,2024-03-01 12:20:18.089,card,38.7,Hot Chocolate,12,4,3,0.940585,1.413261
3,2024-03-01 13:46:33.006,card,28.9,Americano,13,4,3,0.492687,-0.578639
4,2024-03-01 13:48:14.626,card,38.7,Latte,13,4,3,0.940585,1.413261


# Частина 2. Індивідуальні завдання

## Завдання 2. Штучні пропуски та дублікати в датасеті продажів
Завантажуємо датасет заново, додаємо 5 пропусків у `money` та 3 дублікати, потім очищаємо.

In [20]:
sales = pd.read_csv('/content/drive/MyDrive/index_1.csv')
print("Початковий розмір:", sales.shape)

# 5 пропусків у колонці money (seed для відтворюваності)
rng = np.random.default_rng(42)
nan_idx = rng.choice(sales.index, size=5, replace=False)
sales.loc[nan_idx, 'money'] = np.nan

# 3 дублікати: копіюємо 3 випадкові рядки в кінець таблиці
sales = pd.concat([sales, sales.sample(3, random_state=42)], ignore_index=True)

print("Розмір після псування:", sales.shape)
print("Пропусків у money:", sales['money'].isnull().sum())
print("Дублікатів:", sales.duplicated().sum())

Початковий розмір: (3636, 6)
Розмір після псування: (3639, 6)
Пропусків у money: 5
Дублікатів: 3


Очищення: пропуски заповнюємо середнім, дублікати видаляємо, `datetime` перетворюємо на `datetime64`, результат зберігаємо.

In [21]:
sales['money'] = sales['money'].fillna(sales['money'].mean())
sales = sales.drop_duplicates()
sales['datetime'] = pd.to_datetime(sales['datetime'], errors='coerce')

print("Розмір після очищення:", sales.shape)
print("Пропусків у money:", sales['money'].isnull().sum())
print(sales.dtypes)

sales.to_csv('/content/drive/MyDrive/sales_cleaned.csv', index=False)
print("Файл збережено!")

Розмір після очищення: (3636, 6)
Пропусків у money: 0
date                   object
datetime       datetime64[ns]
cash_type              object
card                   object
money                 float64
coffee_name            object
dtype: object
Файл збережено!


**Зауваження:** колонка `card` має 89 пропусків уже в оригінальних даних. Це оплата готівкою, де картки немає, тому ці пропуски не є помилкою й заповнювати їх не потрібно.

# Частина 3. ★ Додаткове завдання: функція `clean_dataset(df)`
Функція повертає очищену копію: видаляє повні дублікати, заповнює пропуски (числові медіаною, категоріальні модою), перетворює колонки з датою на `datetime64` і видаляє рядки з викидами за IQR у всіх числових колонках. Колонки з датою визначаються за словом `date` у назві (можна передати свій список через `date_cols`).

In [22]:
def clean_dataset(data, date_cols=None):
    """Повертає очищену копію DataFrame."""
    result = data.copy()

    # 1. Повні дублікати
    result = result.drop_duplicates()

    # Колонки з датою: за замовчуванням ті, що містять 'date' у назві
    if date_cols is None:
        date_cols = [c for c in result.columns if 'date' in str(c).lower()]

    # 2. Пропуски: числові - медіаною, решта - модою
    for col in result.columns:
        if pd.api.types.is_bool_dtype(result[col]):
            continue
        if pd.api.types.is_numeric_dtype(result[col]):
            result[col] = result[col].fillna(result[col].median())
        else:
            mode = result[col].mode()
            if not mode.empty:
                result[col] = result[col].fillna(mode[0])

    # 3. Колонки з датою -> datetime64
    for col in date_cols:
        result[col] = pd.to_datetime(result[col], errors='coerce')

    # 4. Викиди за IQR у всіх числових колонках
    num_cols = [c for c in result.columns
                if pd.api.types.is_numeric_dtype(result[c])
                and not pd.api.types.is_bool_dtype(result[c])]
    for col in num_cols:
        q1, q3 = result[col].quantile(0.25), result[col].quantile(0.75)
        iqr = q3 - q1
        result = result[(result[col] >= q1 - 1.5 * iqr) & (result[col] <= q3 + 1.5 * iqr)]

    return result.reset_index(drop=True)

Тестуємо на штучному «брудному» датасеті: є пропуски, дублікат, дати текстом і викид (`price` = 500, `qty` = 90).

In [23]:
test = pd.DataFrame({
    'order_date': ['2024-03-01', '2024-03-02', '2024-03-02', '2024-03-03', '2024-03-04',
                   '2024-03-05', '2024-03-06', '2024-03-07', '2024-03-08', '2024-03-02'],
    'product':    ['Latte', 'Cocoa', 'Cocoa', None, 'Latte', 'Latte', 'Espresso', 'Cocoa', 'Latte', 'Cocoa'],
    'price':      [38.7, 35.0, 35.0, 33.0, None, 38.7, 500.0, 30.0, 32.0, 35.0],
    'qty':        [2, 1, 1, 3, 2, None, 2, 90, 1, 1]
})
print("До очищення:", test.shape)
print(test)

cleaned = clean_dataset(test)

print("\nПісля очищення:", cleaned.shape)
print(cleaned)
print(cleaned.dtypes)

До очищення: (10, 4)
   order_date   product  price   qty
0  2024-03-01     Latte   38.7   2.0
1  2024-03-02     Cocoa   35.0   1.0
2  2024-03-02     Cocoa   35.0   1.0
3  2024-03-03      None   33.0   3.0
4  2024-03-04     Latte    NaN   2.0
5  2024-03-05     Latte   38.7   NaN
6  2024-03-06  Espresso  500.0   2.0
7  2024-03-07     Cocoa   30.0  90.0
8  2024-03-08     Latte   32.0   1.0
9  2024-03-02     Cocoa   35.0   1.0

Після очищення: (6, 4)
  order_date product  price  qty
0 2024-03-01   Latte   38.7  2.0
1 2024-03-02   Cocoa   35.0  1.0
2 2024-03-03   Latte   33.0  3.0
3 2024-03-04   Latte   35.0  2.0
4 2024-03-05   Latte   38.7  2.0
5 2024-03-08   Latte   32.0  1.0
order_date    datetime64[ns]
product               object
price                float64
qty                  float64
dtype: object


Зберігаємо результат у CSV.

In [24]:
cleaned.to_csv('/content/drive/MyDrive/cleaned_test.csv', index=False)
print("Файл збережено!")

Файл збережено!
